# Notebook 4: SAC for Intraday Trading - State-of-the-Art RL

## Welcome to Advanced RL! 🚀

You've mastered DQN and PPO. Now let's explore **SAC (Soft Actor-Critic)** - one of the most advanced and effective RL algorithms!

### What you'll learn:
1. **What is SAC?** and why it's state-of-the-art
2. **Maximum Entropy RL** - automatic exploration
3. **Training a SAC agent** for intraday trading
4. **Comparing SAC with DQN and PPO**
5. **Understanding continuous action spaces**

### Algorithm Progression:

| Algorithm | Type | Key Feature | Best For |
|-----------|------|-------------|----------|
| DQN | Value-based | Simple, fast | Discrete actions |
| PPO | Policy gradient | Stable | General purpose |
| **SAC** | **Actor-Critic** | **Auto-tuned exploration** | **Production systems** |

SAC combines the best of both worlds! 💪

## Step 1: Import Libraries

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(ROOT)

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from mmcv import Config

from trademaster.utils import replace_cfg_vals, set_seed
from trademaster.nets.builder import build_net
from trademaster.environments.builder import build_environment
from trademaster.datasets.builder import build_dataset
from trademaster.agents.builder import build_agent
from trademaster.optimizers.builder import build_optimizer
from trademaster.losses.builder import build_loss
from trademaster.trainers.builder import build_trainer
from trademaster.transition.builder import build_transition

set_seed(42)

print("✅ Libraries imported!")
print(f"🔥 PyTorch: {torch.__version__}")
print(f"🎯 CUDA: {torch.cuda.is_available()}")

## Step 2: Understanding SAC

### What makes SAC special?

**Soft Actor-Critic (SAC)** introduces **Maximum Entropy RL**:

1. **Goal**: Maximize reward + entropy (randomness)
2. **Benefit**: Automatic exploration-exploitation balance
3. **Result**: More robust and stable learning

### SAC Key Features:
- ✅ **Off-policy**: Sample efficient (learns from old data)
- ✅ **Entropy tuning**: Automatically balances explore/exploit
- ✅ **Twin critics**: Reduces overestimation bias
- ✅ **Continuous actions**: Perfect for position sizing
- ✅ **State-of-the-art**: Used in robotics, trading, gaming

### Why SAC for Trading?
- **Robust**: Works across different market conditions
- **Sample efficient**: Learns faster from limited data
- **Automatic**: Less hyperparameter tuning needed
- **Production-ready**: Stable and reliable

Let's train it!

## Step 3: Configure SAC

In [ ]:
CONTRACT = 'MCL-1m'

DATA_DIR = Path('./data') / CONTRACT
WORK_DIR = Path('./saved_models') / f'{CONTRACT}_sac'
WORK_DIR.mkdir(parents=True, exist_ok=True)

print(f"🎯 Contract: {CONTRACT}")
print(f"📁 Data: {DATA_DIR}")
print(f"💾 Models: {WORK_DIR}")

In [ ]:
# SAC Configuration
config = {
    'task_name': 'algorithmic_trading',
    'dataset_name': CONTRACT,
    'work_dir': str(WORK_DIR),
    
    'data': {
        'type': 'AlgorithmicTradingDataset',
        'data_path': str(DATA_DIR),
        'train_path': str(DATA_DIR / 'train.csv'),
        'valid_path': str(DATA_DIR / 'valid.csv'),
        'test_path': str(DATA_DIR / 'test.csv'),
        'tech_indicator_list': [
            'high', 'low', 'open', 'close', 'adjcp',
            'zopen', 'zhigh', 'zlow', 'zadjcp', 'zclose',
            'zd_5', 'zd_10', 'zd_15', 'zd_20', 'zd_25', 'zd_30'
        ],
        'backward_num_day': 5,
        'forward_num_day': 5,
        'test_dynamic': '-1'
    },
    
    'environment': {
        'type': 'AlgorithmicTradingEnvironment'
    },
    
    # SAC Agent Configuration
    'agent': {
        'type': 'AlgorithmicTradingSAC',
        'max_step': 10000,
        'reward_scale': 1,
        'gamma': 0.99,
        'batch_size': 128,  # Larger batch for SAC
        'learning_rate': 0.0003,
        'target_entropy': None,  # Auto-tuned!
        'alpha': 0.2,  # Entropy temperature (auto-adjusted)
        'tau': 0.005,  # Soft target update
        'repeat_times': 1,
    },
    
    'trainer': {
        'type': 'AlgorithmicTradingTrainer',
        'epochs': 5,
        'work_dir': str(WORK_DIR),
        'seeds_list': (42,),
        'batch_size': 128,
        'horizon_len': 256,
        'buffer_size': 100000,
        'num_threads': 4,
        'if_remove': False,
        'if_discrete': True,
        'if_off_policy': True,  # SAC is off-policy!
        'if_keep_save': True,
        'if_over_write': False,
        'if_save_buffer': False
    },
    
    'loss': {'type': 'MSELoss'},
    'optimizer': {'type': 'Adam', 'lr': 0.0003},
    
    # SAC Networks (Actor + Twin Critics)
    'act': {
        'type': 'ActorSAC',
        'state_dim': 82,
        'action_dim': 3,
        'dims': (128, 128),  # Larger network
    },
    
    'cri': {
        'type': 'CriticTwin',  # Twin critics reduce overestimation
        'state_dim': 82,
        'action_dim': 3,
        'dims': (128, 128),
    },
    
    'transition': {'type': 'Transition'},
    'batch_size': 128
}

cfg = Config(config)
cfg = replace_cfg_vals(cfg)

print("✅ SAC Configuration created!")
print(f"\n🎯 SAC Key Features:")
print(f"  - Twin Critics: Reduces overestimation")
print(f"  - Alpha (entropy): {cfg.agent['alpha']} (auto-tuned during training)")
print(f"  - Tau (soft update): {cfg.agent['tau']}")
print(f"  - Off-policy: ✅ (sample efficient)")
print(f"  - Batch size: {cfg.batch_size} (larger than DQN/PPO)")

## Step 4: Build SAC Components

In [ ]:
print("🔨 Building SAC components...\n")

print("📊 Dataset...")
dataset = build_dataset(cfg)
print(f"   ✅ {len(dataset.train_df):,} samples")

print("\n🌍 Environments...")
train_env = build_environment(cfg, task='train')
valid_env = build_environment(cfg, task='valid')
print(f"   ✅ Created")

print("\n🎭 Actor (Stochastic Policy)...")
act = build_net(cfg.act)
print(f"   ✅ {sum(p.numel() for p in act.parameters()):,} parameters")

print("\n👥 Twin Critics (Q-functions)...")
cri = build_net(cfg.cri)
print(f"   ✅ {sum(p.numel() for p in cri.parameters()):,} parameters")
print("   💡 Two critics prevent overestimation!")

print("\n⚙️ Optimizers...")
act_optimizer = build_optimizer(cfg, act)
cri_optimizer = build_optimizer(cfg, cri)
criterion = build_loss(cfg)
print(f"   ✅ Ready")

print("\n💾 Replay Buffer...")
transition = build_transition(cfg)
print(f"   ✅ {cfg.trainer['buffer_size']:,.0f} capacity")

print("\n🤖 SAC Agent...")
agent = build_agent(cfg, train_env, act, cri, act_optimizer, cri_optimizer, criterion)
print(f"   ✅ Created with maximum entropy objective!")

print("\n🎓 Trainer...")
trainer = build_trainer(cfg, train_env, valid_env, agent, transition)
print(f"   ✅ Ready!")

print("\n" + "="*60)
print("✅ SAC ready to learn!")
print("="*60)

## Step 5: Train SAC Agent 🚀

SAC training is:
1. **Sample efficient** (off-policy learning)
2. **Stable** (twin critics, entropy tuning)
3. **Automatic** (self-adjusting exploration)

Watch the entropy coefficient adjust automatically! ✨

In [ ]:
print("🚀 Training SAC agent...\n")
print("SAC features automatic entropy tuning for optimal exploration!")
print("This may take 15-40 minutes.")
print("\n" + "="*60)

trainer.train_and_valid()

print("\n" + "="*60)
print("🎉 SAC Training complete!")
print("="*60)

## Step 6: Evaluate SAC

In [ ]:
print("📈 Testing SAC agent...\n")

test_env = build_environment(cfg, task='test')
trainer.test()

print("\n✅ Evaluation complete!")

## Step 7: Visualize SAC Results

In [ ]:
result_path = WORK_DIR / 'test' / f'{CONTRACT}_algorithmic_trading_test.csv'

if result_path.exists():
    results_df = pd.read_csv(result_path)
    
    fig, axes = plt.subplots(3, 1, figsize=(15, 12))
    
    # Portfolio
    axes[0].plot(results_df.index, results_df['portfolio_value'], linewidth=2, color='purple', label='SAC Agent')
    axes[0].axhline(y=test_env.initial_amount, color='red', linestyle='--', label='Initial')
    axes[0].set_title('SAC: Portfolio Value Over Time', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Portfolio Value ($)', fontsize=12)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Actions
    if 'action' in results_df.columns:
        action_colors = {0: 'red', 1: 'gray', 2: 'green'}
        action_labels = {0: 'Sell', 1: 'Hold', 2: 'Buy'}
        for action, color in action_colors.items():
            mask = results_df['action'] == action
            axes[1].scatter(results_df.index[mask], [action]*mask.sum(), 
                          c=color, alpha=0.6, label=action_labels[action], s=10)
        axes[1].set_title('SAC: Trading Actions', fontsize=14, fontweight='bold')
        axes[1].set_ylabel('Action', fontsize=12)
        axes[1].set_yticks([0, 1, 2])
        axes[1].set_yticklabels(['Sell', 'Hold', 'Buy'])
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
    
    # Returns
    if 'portfolio_return' in results_df.columns:
        axes[2].plot(results_df.index, results_df['portfolio_return'], linewidth=1, alpha=0.7, color='purple')
        axes[2].axhline(y=0, color='red', linestyle='--')
        axes[2].set_title('SAC: Returns', fontsize=14, fontweight='bold')
        axes[2].set_xlabel('Time Step', fontsize=12)
        axes[2].set_ylabel('Return', fontsize=12)
        axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Statistics
    print("\n📊 SAC Performance Summary:\n")
    print("="*60)
    final_value = results_df['portfolio_value'].iloc[-1]
    initial_value = test_env.initial_amount
    total_return = (final_value - initial_value) / initial_value * 100
    
    print(f"Initial Capital:       ${initial_value:,.2f}")
    print(f"Final Portfolio Value: ${final_value:,.2f}")
    print(f"Total Return:          {total_return:+.2f}%")
    print(f"Profit/Loss:           ${final_value - initial_value:+,.2f}")
    
    if 'portfolio_return' in results_df.columns:
        returns = results_df['portfolio_return'].dropna()
        if len(returns) > 0:
            sharpe = returns.mean() / (returns.std() + 1e-8) * np.sqrt(252 * 1440)
            print(f"\nSharpe Ratio (Ann):    {sharpe:.2f}")
            print(f"Max Drawdown:          {(results_df['portfolio_value'].min() - initial_value) / initial_value * 100:.2f}%")
    
    print("="*60)
else:
    print(f"⚠️  Results not found")

## Step 8: Compare All Three Algorithms

Let's see how DQN, PPO, and SAC stack up!

In [ ]:
# Load all results
dqn_path = Path('./saved_models') / CONTRACT / 'test' / f'{CONTRACT}_algorithmic_trading_test.csv'
ppo_path = Path('./saved_models') / f'{CONTRACT}_ppo' / 'test' / f'{CONTRACT}_algorithmic_trading_test.csv'
sac_path = result_path

results_available = []
if dqn_path.exists():
    dqn_df = pd.read_csv(dqn_path)
    results_available.append(('DQN', dqn_df, 'green'))
if ppo_path.exists():
    ppo_df = pd.read_csv(ppo_path)
    results_available.append(('PPO', ppo_df, 'blue'))
if sac_path.exists():
    sac_df = pd.read_csv(sac_path)
    results_available.append(('SAC', sac_df, 'purple'))

if len(results_available) > 1:
    fig, axes = plt.subplots(2, 1, figsize=(15, 10))
    
    # Portfolio comparison
    for name, df, color in results_available:
        axes[0].plot(df.index, df['portfolio_value'], label=name, linewidth=2, color=color)
    axes[0].axhline(y=test_env.initial_amount, color='red', linestyle='--', label='Initial', linewidth=1.5)
    axes[0].set_title('Algorithm Comparison: Portfolio Value', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Portfolio Value ($)', fontsize=12)
    axes[0].legend(loc='best', fontsize=11)
    axes[0].grid(True, alpha=0.3)
    
    # Cumulative returns
    for name, df, color in results_available:
        if 'portfolio_return' in df.columns:
            axes[1].plot(df.index, df['portfolio_return'].cumsum(), 
                        label=f'{name} Cumulative', linewidth=2, color=color, alpha=0.7)
    axes[1].set_title('Cumulative Returns Comparison', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Time Step', fontsize=12)
    axes[1].set_ylabel('Cumulative Return', fontsize=12)
    axes[1].legend(loc='best', fontsize=11)
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Comparison table
    print("\n📊 FINAL ALGORITHM COMPARISON:\n")
    print("="*80)
    print(f"{'Metric':<30} ", end='')
    for name, _, _ in results_available:
        print(f"{name:>15} ", end='')
    print()
    print("="*80)
    
    # Returns
    print(f"{'Total Return (%)':<30} ", end='')
    returns_dict = {}
    for name, df, _ in results_available:
        ret = (df['portfolio_value'].iloc[-1] - test_env.initial_amount) / test_env.initial_amount * 100
        returns_dict[name] = ret
        print(f"{ret:>14.2f}% ", end='')
    print()
    
    # Final value
    print(f"{'Final Value ($)':<30} ", end='')
    for name, df, _ in results_available:
        print(f"{df['portfolio_value'].iloc[-1]:>15,.2f} ", end='')
    print()
    
    # Sharpe
    print(f"{'Sharpe Ratio':<30} ", end='')
    sharpe_dict = {}
    for name, df, _ in results_available:
        if 'portfolio_return' in df.columns:
            sharpe = df['portfolio_return'].mean() / (df['portfolio_return'].std() + 1e-8) * np.sqrt(252*1440)
            sharpe_dict[name] = sharpe
            print(f"{sharpe:>15.2f} ", end='')
    print()
    
    print("="*80)
    
    # Determine winner
    best_algo = max(returns_dict, key=returns_dict.get)
    print(f"\n🏆 Winner (by return): {best_algo}")
    if sharpe_dict:
        best_sharpe = max(sharpe_dict, key=sharpe_dict.get)
        print(f"🏆 Winner (by Sharpe): {best_sharpe}")
    
    print("\n💡 Key Insights:")
    print("  - SAC usually has best risk-adjusted returns (Sharpe)")
    print("  - DQN is fastest but can be less stable")
    print("  - PPO is most consistent across runs")
    print("  - Results vary by market conditions and hyperparameters!")
    
else:
    print("⚠️  Run other notebooks to compare algorithms!")

## Summary: Your RL Journey

### 🎉 Congratulations!

You've mastered the three most important RL algorithms for trading!

### 🧠 What You've Learned:

#### DQN (Deep Q-Network):
- ✅ Value-based learning
- ✅ Fast training
- ✅ Good for discrete actions
- ⚠️ Can be unstable

#### PPO (Proximal Policy Optimization):
- ✅ Policy gradient method
- ✅ Very stable
- ✅ Works with continuous actions
- ✅ Industry standard

#### SAC (Soft Actor-Critic):
- ✅ Maximum entropy RL
- ✅ Auto-tuned exploration
- ✅ State-of-the-art performance
- ✅ Sample efficient
- ✅ Production-ready

### 🎯 Which Algorithm Should You Use?

| Situation | Best Choice | Why |
|-----------|-------------|-----|
| Quick prototyping | **DQN** | Fast, simple |
| Production system | **SAC** | Robust, automatic |
| Limited compute | **PPO** | Stable, efficient |
| Complex strategies | **SAC/PPO** | Continuous actions |
| Risk management | **SAC** | Better risk-adjusted returns |

### 💡 Pro Tips:
1. **Start with DQN** to understand basics
2. **Use PPO** for stable baseline
3. **Deploy SAC** for production
4. **Ensemble** multiple algorithms for robustness
5. **Always backtest** on multiple time periods

### 🚀 Next Steps:
In **Notebook 5**, you'll learn:
- Advanced technical indicators
- Risk management strategies
- Position sizing
- Portfolio optimization
- Real-world deployment tips

Ready to level up? Let's go! 🎯

---
**Final Pro Tips**:
- **SAC** is the go-to for real trading systems
- Combine RL with **traditional indicators** (next notebook!)
- Always use **risk management** (stop-loss, position limits)
- Paper trade before live trading! 💡